# Multi-size BET pass — BTN vs BB, headless GPU

Re-solves the launch **BTN-vs-BB** single-raised pot across all 17 boards with **two bet sizes**
(`--bet-sizes 0.33,0.75`) instead of the single 66% bet. First-to-act and checked-to spots
become **Check / Bet 33% / Bet 75%** (actions `check`, `bet_33`, `bet_75`), and every
facing-a-bet spot is solved **per size** — facing a 33% bet (`bb_vs_bet_33`, `btn_vs_bet_33`)
and facing a 75% bet (`bb_vs_bet_75`, `btn_vs_bet_75`) are separate Fold/Call spots, and each
record carries `facing_bet_frac`.

No raises in this pass: the 2-size tree costs about the same as the single-size raise tree
(5 lines per street), while 2 sizes + raise is ~6x bigger again — too slow and too
memory-hungry for a T4 commit.

**Per commit:** set `PART='A'` (boards 0–8), then `'B'` (9–16), and Save & Run All each time
(Accelerator → GPU T4 x2, Internet On). Download `records_betsizes_A.json` and
`records_betsizes_B.json` and send both back — they merge board-wise and build/sign a
`betsizes` pack locally.

*The trainer UI does not render per-size buttons yet — this pack is content-first; the app
wiring follows once the records exist (see runbook §3).*

In [ ]:
# Fail fast if no GPU.
import subprocess
try:
    _r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
except FileNotFoundError:
    raise SystemExit('No nvidia-smi (CPU instance) -- set Accelerator -> GPU, then re-run.')
assert _r.returncode == 0 and 'GPU' in _r.stdout, 'NO GPU -- set Accelerator -> GPU T4 x2, then re-run.'
print(_r.stdout.strip())

In [ ]:
# Clone the solver source (needs Internet On) -- pulls BTN-vs-BB + 17 boards + bet_fracs support.
!rm -rf /kaggle/working/poker && git clone -q --depth 1 https://github.com/tian-chaiyaporn2/poker_offline_trainer /kaggle/working/poker
import sys; sys.path.insert(0, '/kaggle/working/poker/src')
from pokertrainer.presets import SCENARIOS, BOARDS
BET_SIZES = '0.33,0.75'
print('scenario:', SCENARIOS['btn_vs_bb_srp']['label'], '| boards:', len(BOARDS), '| bet sizes:', BET_SIZES)

In [ ]:
try:
    import cupy as cp
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x']); import cupy as cp
free,total=cp.cuda.Device(0).mem_info
nm=cp.cuda.runtime.getDeviceProperties(0)['name']; print('cupy',cp.__version__,'|',nm.decode() if isinstance(nm,bytes) else nm)
print('GPU memory: %.1f GB free / %.1f GB total'%(free/1e9,total/1e9))

In [ ]:
# Fail-fast smoke (seconds): the multi-size tree on the CuPy backend stays float32 and emits
# per-size labels/nodes. Catches a stale clone (no bet_fracs) before a multi-hour commit.
import numpy as np
from pokertrainer.cards import parse_cards, parse_hand
from pokertrainer.solver.batched_gpu import BatchedGPUCFR
_f = parse_cards('As7h2d')
_o = [parse_hand(h) for h in ['AhAc', 'KsKc', '7s7c', 'AhKh', 'Ts9s', 'QsQd']]
_i = [parse_hand(h) for h in ['AhQh', 'KsKh', 'JsJh', 'AhTh', 'Tc9c', '9s8s']]
_s = BatchedGPUCFR(_f, _o, _i, np.ones(6), np.ones(6), 5.5, 0.66, streets=3, backend='cupy',
                   dtype='float32', bet_fracs=[float(x) for x in BET_SIZES.split(',')])
_s.run(2)
assert _s.backend == 'cupy', 'solver fell back to NumPy -- CuPy is broken'
assert all(str(a.dtype) == 'float32' for a in list(_s.R.values()) + list(_s.S.values())), 'float64 leak'
_recs = _s.flop_decisions_report()
_nodes = sorted({r['node'] for r in _recs})
assert {'bb_vs_bet_33', 'bb_vs_bet_75', 'btn_vs_bet_33', 'btn_vs_bet_75'} <= set(_nodes), _nodes
assert all(r['actions'] == ['check', 'bet_33', 'bet_75'] for r in _recs if r['node'] == 'bb_first')
print('smoke OK | nodes:', _nodes)
del _s, _recs

## Solve one part, then Save & Run All. Recommended: **2 parts** (`PART='A'` then `'B'`).

Set `SPLIT=2` and run `PART='A'` (boards 0-8), then `PART='B'` (9-16) in a second commit
— each ~2-4.5 h (same cost class as the single-size raise pass), safely under Kaggle's ~9 h
limit. If a part ever runs long, drop to `SPLIT=3`, or split its `ROOTS` by hand. The merge
is board-wise, so any grouping recombines cleanly.

In [ ]:
# --- Pick a split. The 2-size tree is the same size as the single-size raise tree, so a
# 2-part split (~2-4.5 h each) stays well under Kaggle's ~9 h commit limit. ---
SPLIT = 2        # 2 = two commits (recommended); 3 = three shorter commits
PART  = 'A'      # 2-part: 'A' (boards 0-8), 'B' (9-16)
                 # 3-part: 'A' (0-5), 'B' (6-11), 'C' (12-16)
assert SPLIT in (2, 3), 'SPLIT must be 2 or 3'
assert PART in {2: ('A', 'B'), 3: ('A', 'B', 'C')}[SPLIT], f'PART {PART!r} is not valid for SPLIT={SPLIT}'
ROOTS = ({2: {'A': '0,1,2,3,4,5,6,7,8', 'B': '9,10,11,12,13,14,15,16'},
          3: {'A': '0,1,2,3,4,5', 'B': '6,7,8,9,10,11', 'C': '12,13,14,15,16'}}[SPLIT])[PART]
import subprocess, os
env = {**os.environ, 'PYTHONPATH': 'src'}
print(f'SPLIT={SPLIT} PART={PART} -> boards {ROOTS} | bet sizes {BET_SIZES}')
subprocess.run(
    ['python', '-m', 'pokertrainer.content_yield',
     '--solver', 'gpu', '--dtype', 'float32', '--n', '400', '--iters', '300',
     '--scenario', 'btn_vs_bb_srp', '--bet-sizes', BET_SIZES, '--roots', ROOTS,
     '--out', f'/kaggle/working/cy_betsizes_{PART}'],
    cwd='/kaggle/working/poker', env=env, check=True)

In [ ]:
# Expose records_betsizes_<PART>.json for download (fails loudly if nothing completed).
import json, shutil, os
from collections import Counter
try:
    rep = json.load(open(f'/kaggle/working/cy_betsizes_{PART}/yield_report.json'))
except (FileNotFoundError, json.JSONDecodeError):
    raise SystemExit(f'No cy_betsizes_{PART} output -- check the cell above and cy_betsizes_{PART}/boards/*.ERROR.txt')
done = rep.get('boards_completed') or []
print('PART', PART, '| boards completed:', done, '| missing:', rep.get('boards_missing'))
if not done:
    raise SystemExit(f'No boards completed in PART {PART} -- check cy_betsizes_{PART}/boards/*.ERROR.txt')
recs = json.load(open(f'/kaggle/working/cy_betsizes_{PART}/records.json'))
sized = sum(1 for r in recs if any(a.startswith('bet_') for a in r.get('actions') or []))
if not sized:
    raise SystemExit('records carry no bet_<size> actions -- the multi-size flag did not take effect')
shutil.copy(f'/kaggle/working/cy_betsizes_{PART}/records.json', f'/kaggle/working/records_betsizes_{PART}.json')
print('per node:', dict(Counter(r['node'] for r in recs)))
print('DOWNLOAD -> records_betsizes_%s.json (%d KB, %d records, %d with sized bets)'
      % (PART, os.path.getsize(f'/kaggle/working/records_betsizes_{PART}.json')//1024, len(recs), sized))